# Vehicle Loan Default Prediction Pipeline\nThis notebook implements a cost-sensitive, calibrated stacking ensemble to predict vehicle loan defaults.

## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import category_encoders as ce

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, 
    fbeta_score, brier_score_loss, precision_recall_curve
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

### 1.1 Compatibility Checks

In [ ]:
# Handle deprecation of cv='prefit' in newer sklearn versions
try:
    from sklearn.calibration import FrozenEstimator
    USE_FROZEN = True
except ImportError:
    USE_FROZEN = False

## 2. Data Loading

In [ ]:
# Load processed chunks
df_loan = pd.read_csv('../dataset/processed/loan_details.csv')
df_customer = pd.read_csv('../dataset/processed/customer_profile.csv')
df_bureau = pd.read_csv('../dataset/processed/bureau_history.csv')
df_risk = pd.read_csv('../dataset/processed/engineered_risk.csv')

### 2.1 Merging Datasets

In [ ]:
# Merge into single analytical base table (ABT)
df = df_loan.merge(df_customer, on='customer_id')\
            .merge(df_bureau, on='customer_id')\
            .merge(df_risk, on='customer_id')

### 2.2 Temporal Features

In [ ]:
# Extract temporal features from raw data
df_raw = pd.read_csv('../dataset/train.csv')
df_raw['DisbursalDate'] = pd.to_datetime(df_raw['DisbursalDate'], format='%d-%m-%y', errors='coerce')

df['Disbursal_Quarter'] = df_raw['DisbursalDate'].dt.quarter.values
df['Disbursal_Month'] = df_raw['DisbursalDate'].dt.month.values
df['Is_End_Of_Month'] = df_raw['DisbursalDate'].dt.is_month_end.astype(int).values

### 2.3 Categorical Encoding (Basic)

In [ ]:
# One-hot encode employment type
df = pd.get_dummies(df, columns=['employment_type'], drop_first=True, dtype=int)

## 3. Feature Engineering

### 3.1 Pruning

In [ ]:
# Drop low-variance or highly collinear features
cols_to_drop = [
    'downpayment_ratio', 'pri_account_count', 'pri_active_accounts', 
    'pri_overdue_accounts', 'pri_current_balance', 'pri_sanctioned_amount', 
    'pri_disbursed_amount', 'pri_installment_amount', 'has_credit_history', 
    'has_passport', 'has_driving_license', 'credit_hunger_delta', 'has_voter_id'
]
df = df.drop(columns=cols_to_drop, errors='ignore')

### 3.2 Interaction Features

In [ ]:
# Engineer interaction features based on financial logic
df['ltv_x_bureau'] = df['ltv'] * df['bureau_score']
df['installment_burden'] = df['total_installment_amount'] / (df['disbursed_amount'] + 1)
df['overdue_per_account'] = df['total_overdue_accounts'] / (df['total_account_count'] + 1)
df['balance_to_sanctioned'] = df['total_current_balance'] / (df['total_sanctioned_amount'] + 1)
df['inquiry_per_month'] = df['inquiry_count'] / (df['credit_history_months'] + 1)
df['age_x_credit'] = df['age_at_disbursal'] * df['credit_history_months']
df['new_acct_velocity'] = df['new_accounts_6m'] / (df['total_account_count'] + 1)

## 4. Preprocessing Configuration

### 4.1 Define Target & Features

In [ ]:
X = df.drop(columns=['loan_default', 'asset_cost', 'customer_id'], errors='ignore')
y = df['loan_default']

### 4.2 Format High-Cardinality Categoricals

In [ ]:
cat_cols = ['branch_id', 'supplier_id', 'manufacturer_id', 'pincode_id', 'state_id', 'employee_id']
for c in cat_cols:
    if c in X.columns:
        X[c] = X[c].fillna(-1).astype(int).astype(str)

### 4.3 Log-Transform Skewed Features

In [ ]:
# Log-transform highly skewed currency features to pull in long tails
skewed_cols = ['disbursed_amount', 'total_current_balance', 'total_sanctioned_amount']
for c in skewed_cols:
    if c in X.columns:
        X[c] = np.log1p(np.clip(X[c], 0, None))

### 4.4 Define Monotonic Constraints

In [ ]:
# Define monotonic constraints to force logical splits in trees
mono_constraints = {
    'ltv': 1,              
    'bureau_score': -1,   
    'total_active_accounts': -1,
    'overdue_ratio': 1,   
    'credit_utilization': 1,
    'age_at_disbursal': -1,
    'inquiry_count': 1
}

### 4.5 Class Imbalance Baseline

In [ ]:
# Calculate baseline weight for class imbalance
base_scale_pos_weight = (len(y) - sum(y)) / sum(y)
print(f"Base scale_pos_weight: {base_scale_pos_weight:.2f}")

## 5. Model Initialization\nDefining the base models. Note: Built-in balancing parameters are removed because we will manually supply dynamic sample weights during training.

In [ ]:
models = {
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        monotone_constraints=mono_constraints, subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, eval_metric='auc', random_state=42
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, n_jobs=-1, verbose=-1, random_state=42
    ),
    'CatBoost': CatBoostClassifier(
        iterations=300, depth=5, learning_rate=0.05,
        eval_metric='AUC', random_seed=42, verbose=0
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_leaf=50,
        n_jobs=-1, random_state=42
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=200, max_depth=15, min_samples_leaf=50,
        n_jobs=-1, random_state=42
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42
    ),
}

## 6. Helper Functions

In [ ]:
def sweep_f2(y_true, y_pred_proba):
    """Finds the optimal probability threshold to maximize F2 score."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
    denominator = 4 * precision + recall
    f2_scores = np.divide(5 * precision * recall, denominator, out=np.zeros_like(denominator), where=denominator!=0)
    best_idx = np.argmax(f2_scores)
    
    best_f2 = f2_scores[best_idx]
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 1.0
    return best_f2, best_thresh

## 7. Cross-Validation Setup

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = {name: np.zeros(len(X)) for name in models}

### 7.1 Execution Loop\nExecutes a 5-fold Stratified CV. Inside each fold:\n1. Nests a calibration split (80/20).\n2. Applies Target Encoding without leakage.\n3. Computes normalized, cost-sensitive sample weights based on loan size.\n4. Trains models and fits an Isotonic calibrator.

In [ ]:
print("Initiating 5-Fold Training & Calibration...")

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    print(f"\n--- Fold {fold_idx + 1}/5 ---")
    
    X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx].copy(), y.iloc[val_idx]
    
    # 1. Nested split for calibration (stratified to maintain default ratio)
    X_t, X_cal, y_t, y_cal = train_test_split(
        X_train, y_train, test_size=0.20, stratify=y_train, random_state=42
    )
    
    # 2. Target encoding (fit on training subset only)
    te = ce.TargetEncoder(cols=cat_cols, smoothing=10, min_samples_leaf=20)
    X_t = te.fit_transform(X_t, y_t).fillna(0)
    X_cal = te.transform(X_cal).fillna(0)
    X_val = te.transform(X_val).fillna(0)
    
    # 3. Compute normalized sample weights (penalize missing large loans)
    mean_disbursed = np.expm1(X_t['disbursed_amount']).mean()
    fin_penalty = np.clip(np.expm1(X_t['disbursed_amount']) / mean_disbursed, 0.5, 3.0)
    base_weight = np.where(y_t == 1, base_scale_pos_weight, 1.0)
    
    sample_weight_t = base_weight * fin_penalty
    sample_weight_t = sample_weight_t / sample_weight_t.mean() 
    
    for model_name, model_template in models.items():
        model = clone(model_template)
        
        try:
            # 4. Train with weights
            model.fit(X_t, y_t, sample_weight=sample_weight_t)
            
            # 5. Calibrate without weights
            if USE_FROZEN:
                calibrator = CalibratedClassifierCV(FrozenEstimator(model), method='isotonic')
            else:
                calibrator = CalibratedClassifierCV(model, cv='prefit', method='isotonic')
            
            calibrator.fit(X_cal, y_cal)
            
            # 6. Predict
            preds = calibrator.predict_proba(X_val)[:, 1]
            oof_preds[model_name][val_idx] = preds
            
            # Metrics
            f_pr = average_precision_score(y_val, preds)
            f_f2, f_thresh = sweep_f2(y_val, preds)
            print(f"  {model_name.ljust(16)} | PR: {f_pr:.3f} | F2: {f_f2:.3f} (@{f_thresh:.2f})")
            
        except Exception as e:
            oof_preds[model_name][val_idx] = 0.5
            print(f"  {model_name} failed: {str(e)[:50]}")

## 8. Base Model Evaluation

In [ ]:
oof_scores = {}
print("Base Models (OOF Performance):")

for name in models:
    try:
        preds = oof_preds[name]
        r_roc = roc_auc_score(y, preds)
        r_pr = average_precision_score(y, preds)
        r_f2, r_thresh = sweep_f2(y, preds)
        r_brier = brier_score_loss(y, preds)
        oof_scores[name] = {'ROC': r_roc, 'PR': r_pr, 'F2': r_f2, 'F2_Thresh': r_thresh, 'Brier': r_brier}
        print(f"{name.ljust(18)} PR-AUC: {r_pr:.4f} | F2: {r_f2:.4f} (@{r_thresh:.2f}) | Brier: {r_brier:.4f}")
    except:
        pass

### 8.1 Select Top Models

In [ ]:
# Select Top 3 for Stacking
leaderboard = sorted(oof_scores.items(), key=lambda x: x[1]['PR'], reverse=True)
top3_names = [name for name, _ in leaderboard[:3]]
print(f"Selected for Stacking: {top3_names}")

## 9. Base Model Calibration Curve

In [ ]:
os.makedirs('../charts', exist_ok=True)

plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")

for name in top3_names:
    prob_true, prob_pred = calibration_curve(y, oof_preds[name], n_bins=10, strategy='quantile')
    plt.plot(prob_pred, prob_true, marker='o', label=name)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Actual Fraction of Defaults")
plt.title("Base Models Reliability Diagram")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig('../charts/12a_base_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Meta-Model Stacking\nBlending the calibrated probabilities using Logistic Regression to find the optimal ensemble weights.

In [ ]:
meta_X = np.column_stack([oof_preds[name] for name in top3_names])
meta_model = LogisticRegression(max_iter=1000, random_state=42)
meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

meta_preds = np.zeros(len(X))
for train_idx, val_idx in meta_cv.split(meta_X, y):
    meta_model.fit(meta_X[train_idx], y.iloc[train_idx])
    meta_preds[val_idx] = meta_model.predict_proba(meta_X[val_idx])[:, 1]

### 10.1 Meta-Model Evaluation

In [ ]:
s_roc = roc_auc_score(y, meta_preds)
s_pr = average_precision_score(y, meta_preds)
s_f2, s_thresh = sweep_f2(y, meta_preds)
s_brier = brier_score_loss(y, meta_preds)

print("Stacked Meta-Model Performance:")
print(f"PR-AUC:   {s_pr:.4f}")
print(f"F2-Score: {s_f2:.4f} (@{s_thresh:.2f})")
print(f"Brier:    {s_brier:.4f}")
print(f"ROC-AUC:  {s_roc:.4f}")

### 10.2 Meta-Model Weights

In [ ]:
# Train final meta-model on all data to inspect weights
meta_model.fit(meta_X, y)
print("Ensemble Weights:")
for name, coef in zip(top3_names, meta_model.coef_[0]):
    print(f"  {name}: {coef:.4f}")

## 11. Meta-Model Calibration Curve

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
prob_true, prob_pred = calibration_curve(y, meta_preds, n_bins=10, strategy='quantile')
plt.plot(prob_pred, prob_true, marker='s', color='purple', label='Stacked Meta-Model')

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Actual Fraction of Defaults")
plt.title("Stacked Meta-Model Reliability Diagram")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig('../charts/12b_meta_calibration.png', dpi=150, bbox_inches='tight')
plt.show()